# ANÁLISE DE DUPLA MASSA

Notebook independente para análise de consistência por **dupla massa** de cada estação pluviométrica.

Fluxo por estação:

1. recupera apenas os apoios permitidos em `estacoes_apoio.csv`;
2. conta a disponibilidade efetiva de cada apoio no arquivo mensal preenchido;
3. ordena por maior número de dados válidos e aplica os desempates configurados;
4. seleciona no máximo 3 apoios;
5. constrói um único período comum por `dropna()` simultâneo em **alvo + apoios selecionados**;
6. calcula a média regional mensal somente com os apoios;
7. acumula a precipitação regional e a precipitação da estação alvo;
8. ajusta a regressão `Y = a + bX` nos acumulados;
9. recalcula o R² da curva de dupla massa;
10. produz CSVs de auditoria e gráficos Plotly individuais e interativo com seletor de estação.

Nenhuma estação fora da lista permitida de apoios é inventada ou utilizada.


## 1. Configuração

In [1]:

from pathlib import Path
import os
import csv
import json
import math
import shutil
import warnings

import numpy as np
import pandas as pd
import yaml
import plotly.graph_objects as go
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

CONFIG_PATH = Path(os.environ.get(
    "CONFIG_DUPLA_MASSA",
    Path.cwd() / "config_dupla_massa.yml"
))

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Configuração não encontrada: {CONFIG_PATH}\n"
        "Coloque config_dupla_massa.yml na mesma pasta do notebook "
        "ou defina a variável de ambiente CONFIG_DUPLA_MASSA."
    )

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

dm = cfg["dupla_massa"]
if not dm.get("enabled", True):
    raise RuntimeError("dupla_massa.enabled está false.")

PATH_DADOS = Path(dm["arquivo_entrada"]).expanduser()
PATH_APOIO = Path(dm["arquivo_estacoes_apoio"]).expanduser()
OUTPUT_DIR = Path(dm["diretorio_saida"]).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CFG = dm["saida"]
INDIV_DIR = OUTPUT_DIR / OUT_CFG["pasta_graficos_individuais"]
if dm["grafico"].get("salvar_individuais_html", True):
    INDIV_DIR.mkdir(parents=True, exist_ok=True)

MAX_APOIOS = int(dm["max_estacoes_apoio"])
MIN_APOIOS = int(dm["min_estacoes_apoio"])
MIN_COMUM = int(dm["min_meses_periodo_comum"])
CASAS = int(dm.get("casas_decimais_equacao", 4))

if MAX_APOIOS < 1:
    raise ValueError("max_estacoes_apoio deve ser >= 1.")
if MIN_APOIOS < 1 or MIN_APOIOS > MAX_APOIOS:
    raise ValueError("min_estacoes_apoio deve estar entre 1 e max_estacoes_apoio.")
if dm.get("criterio_selecao_apoio") != "maior_quantidade_dados":
    raise ValueError("Este notebook implementa criterio_selecao_apoio = maior_quantidade_dados.")
if dm["regional"].get("metodo") != "media_aritmetica":
    raise ValueError("Este notebook implementa regional.metodo = media_aritmetica.")
if not dm.get("exigir_periodo_comum", True):
    raise ValueError("A análise exige exigir_periodo_comum = true.")

print("Configuração:", CONFIG_PATH)
print("Dados:", PATH_DADOS)
print("Apoios:", PATH_APOIO)
print("Saída:", OUTPUT_DIR)


Configuração: a:\Codigo\UNIPLU-BR\2. Analise_basin\config_dupla_massa.yml
Dados: A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\preenchimento_falhas\precipitacao_mensal_regressao_linear.csv
Apoios: A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\estacoes_apoio.csv
Saída: A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\dupla_massa


## 2. Carregamento dos dados preenchidos

In [2]:

DADOS_CFG = dm["dados"]
SEP = DADOS_CFG.get("separador", ";")
DECIMAL = DADOS_CFG.get("decimal", ".")
ENCODING = DADOS_CFG.get("encoding", "utf-8-sig")
COL_DATA = DADOS_CFG.get("coluna_data", "data")
FMT_DATA = DADOS_CFG.get("formato_data", "%Y-%m")
REP_NAN = DADOS_CFG.get("representacao_nan", "NaN")

if not PATH_DADOS.exists():
    raise FileNotFoundError(f"Arquivo principal não encontrado: {PATH_DADOS}")

def ler_cabecalho(path, sep, encoding):
    with open(path, "r", encoding=encoding, newline="") as f:
        primeira = f.readline().rstrip("\r\n")
    return next(csv.reader([primeira], delimiter=sep))

cabecalho = ler_cabecalho(PATH_DADOS, SEP, ENCODING)
duplicadas = sorted({c for c in cabecalho if cabecalho.count(c) > 1})
if duplicadas:
    raise ValueError(f"Colunas duplicadas no arquivo de precipitação: {duplicadas}")

dados_raw = pd.read_csv(
    PATH_DADOS,
    sep=SEP,
    decimal=DECIMAL,
    encoding=ENCODING,
    dtype={COL_DATA: "string"},
    na_values=["NaN", "nan", "NA", "N/A", "null", "None", ""],
    keep_default_na=True,
)

dados_raw.columns = [str(c) for c in dados_raw.columns]

if COL_DATA not in dados_raw.columns:
    raise ValueError(f"Coluna temporal '{COL_DATA}' não encontrada.")

datas = pd.to_datetime(
    dados_raw[COL_DATA].astype("string"),
    format=FMT_DATA,
    errors="coerce"
)

if datas.isna().any():
    exemplos = dados_raw.loc[datas.isna(), COL_DATA].astype(str).head(20).tolist()
    raise ValueError(f"Datas inválidas encontradas: {exemplos}")

if datas.duplicated().any():
    exemplos = datas[datas.duplicated(keep=False)].dt.strftime("%Y-%m").unique().tolist()[:30]
    raise ValueError(f"Datas duplicadas encontradas: {exemplos}")

estacoes = [c for c in dados_raw.columns if c != COL_DATA]
erros_nao_numericos = {}

for c in estacoes:
    original = dados_raw[c]
    conv = pd.to_numeric(original, errors="coerce")
    mask = original.notna() & conv.isna()
    if mask.any():
        erros_nao_numericos[c] = original.loc[mask].astype(str).head(20).tolist()
    dados_raw[c] = conv

if erros_nao_numericos:
    raise ValueError(
        "Valores não numéricos encontrados nas séries: "
        + json.dumps(erros_nao_numericos, ensure_ascii=False)
    )

dados = dados_raw.drop(columns=[COL_DATA]).copy()
dados.index = pd.DatetimeIndex(datas).to_period("M").to_timestamp(how="start")
dados.index.name = COL_DATA
dados = dados.sort_index()

if not dados.index.is_monotonic_increasing:
    raise AssertionError("Falha ao ordenar cronologicamente os dados.")

print(f"Meses carregados: {len(dados):,}")
print(f"Estações carregadas: {len(dados.columns):,}")
print(f"Período: {dados.index.min():%Y-%m} a {dados.index.max():%Y-%m}")


Meses carregados: 540
Estações carregadas: 19
Período: 1980-01 a 2024-12


## 3. Carregamento das estações de apoio

In [3]:

if not PATH_APOIO.exists():
    raise FileNotFoundError(f"Arquivo de apoios não encontrado: {PATH_APOIO}")

apoio = pd.read_csv(
    PATH_APOIO,
    sep=SEP,
    decimal=DECIMAL,
    encoding=ENCODING,
    dtype={
        "estacao_alvo": "string",
        "estacao_apoio": "string",
    }
)

obrigatorias = {
    "estacao_alvo",
    "ranking",
    "estacao_apoio",
    "correlacao_r",
    "r2",
    "meses_coincidentes",
}
faltantes = sorted(obrigatorias - set(apoio.columns))
if faltantes:
    raise ValueError(f"Colunas obrigatórias ausentes em estacoes_apoio.csv: {faltantes}")

apoio["estacao_alvo"] = apoio["estacao_alvo"].astype("string").str.strip()
apoio["estacao_apoio"] = apoio["estacao_apoio"].astype("string").str.strip()
apoio["ranking"] = pd.to_numeric(apoio["ranking"], errors="coerce")
apoio["correlacao_r"] = pd.to_numeric(apoio["correlacao_r"], errors="coerce")
apoio["r2"] = pd.to_numeric(apoio["r2"], errors="coerce")
apoio["meses_coincidentes"] = pd.to_numeric(apoio["meses_coincidentes"], errors="coerce")

if apoio["ranking"].isna().any():
    raise ValueError("Existem rankings não numéricos ou ausentes.")
if (apoio["ranking"] < 1).any() or ((apoio["ranking"] % 1) != 0).any():
    raise ValueError("Todos os rankings devem ser inteiros positivos.")

apoio["ranking"] = apoio["ranking"].astype(int)

# Lista de alvos na ordem em que aparecem no arquivo de apoio.
alvos = [str(x) for x in apoio["estacao_alvo"].dropna().drop_duplicates().tolist()]

print(f"Estações alvo cadastradas: {len(alvos)}")
display(apoio.head())


Estações alvo cadastradas: 19


,estacao_alvo,ranking,estacao_apoio,correlacao_r,r2,meses_coincidentes
0,2751007,1,2751017,0.900560,0.811008,507
1,2751007,2,2852046,0.898935,0.808084,488
2,2751007,3,2851043,0.878377,0.771546,521
3,2751007,4,2851020,0.842247,0.709379,491
4,2751007,5,83914,0.818495,0.669934,449


## 4. Contagem de disponibilidade das estações

In [4]:

disponibilidade_estacoes = dados.notna().sum().astype(int).rename("n_dados_validos").to_frame()
disponibilidade_estacoes.index.name = "estacao"
display(disponibilidade_estacoes.sort_values("n_dados_validos", ascending=False).head(20))

def candidatos_para_alvo(alvo):
    """
    Retorna apoios permitidos com disponibilidade efetiva.
    Critério principal: maior n_dados_validos.
    Desempates: menor ranking, maior correlacao_r, código da estação.
    """
    sub = apoio.loc[apoio["estacao_alvo"].astype(str) == str(alvo)].copy()

    if sub.empty:
        return sub.assign(
            existe_no_arquivo=False,
            n_dados_validos=0,
            apoio_igual_alvo=False
        )

    sub["estacao_apoio"] = sub["estacao_apoio"].astype(str)
    sub["existe_no_arquivo"] = sub["estacao_apoio"].isin(dados.columns)
    sub["apoio_igual_alvo"] = sub["estacao_apoio"].eq(str(alvo))

    sub["n_dados_validos"] = sub["estacao_apoio"].map(
        lambda e: int(dados[e].notna().sum()) if e in dados.columns else 0
    )

    # Não utiliza apoio inexistente, igual ao alvo ou sem nenhuma observação.
    validos = sub[
        sub["existe_no_arquivo"]
        & (~sub["apoio_igual_alvo"])
        & (sub["n_dados_validos"] > 0)
    ].copy()

    # Se houver duplicidade do mesmo par alvo-apoio, ela será tratada como
    # inconsistência específica da estação na etapa de seleção/análise.
    validos = validos.sort_values(
        by=["n_dados_validos", "ranking", "correlacao_r", "estacao_apoio"],
        ascending=[False, True, False, True],
        kind="stable",
        na_position="last"
    )
    return validos.reset_index(drop=True)

# Diagnóstico visual da primeira estação, se houver.
if alvos:
    print("Exemplo de candidatos ordenados para:", alvos[0])
    display(candidatos_para_alvo(alvos[0]))


,n_dados_validos
estacao,
2751007,540
2851003,540
2851022,540
2852046,540
2851021,540
2851024,540
2852004,540
2751017,539
2851043,536


Exemplo de candidatos ordenados para: 2751007


,estacao_alvo,ranking,estacao_apoio,correlacao_r,r2,meses_coincidentes,existe_no_arquivo,apoio_igual_alvo,n_dados_validos
0,2751007,2,2852046,0.898935,0.808084,488,True,False,540
1,2751007,1,2751017,0.900560,0.811008,507,True,False,539
2,2751007,3,2851043,0.878377,0.771546,521,True,False,536
3,2751007,4,2851020,0.842247,0.709379,491,True,False,526
4,2751007,6,2851028,0.810655,0.657162,487,True,False,499
5,2751007,5,83914,0.818495,0.669934,449,True,False,497


## 5. Seleção dos apoios

In [5]:

def selecionar_apoios(alvo):
    sub_all = apoio.loc[apoio["estacao_alvo"].astype(str) == str(alvo)].copy()

    # Inconsistências que tornam a seleção ambígua para este alvo.
    dup_par = sub_all.duplicated(["estacao_alvo", "estacao_apoio"], keep=False)
    if dup_par.any():
        return {
            "ok": False,
            "motivo": "pares_alvo_apoio_duplicados",
            "candidatos": candidatos_para_alvo(alvo),
            "selecionados": pd.DataFrame(),
        }

    dup_rank = sub_all.duplicated(["estacao_alvo", "ranking"], keep=False)
    if dup_rank.any():
        return {
            "ok": False,
            "motivo": "rankings_duplicados",
            "candidatos": candidatos_para_alvo(alvo),
            "selecionados": pd.DataFrame(),
        }

    cand = candidatos_para_alvo(alvo)

    if len(cand) < MIN_APOIOS:
        return {
            "ok": False,
            "motivo": "apoios_disponiveis_insuficientes",
            "candidatos": cand,
            "selecionados": cand.head(0),
        }

    sel = cand.head(MAX_APOIOS).copy()

    return {
        "ok": True,
        "motivo": "ok",
        "candidatos": cand,
        "selecionados": sel,
    }


## 6. Construção do período comum

In [6]:

def construir_periodo_comum(alvo, estacoes_apoio):
    cols = [str(alvo)] + [str(e) for e in estacoes_apoio]
    grupo = dados[cols].copy()
    grupo_comum = grupo.dropna(how="any").sort_index()
    return grupo, grupo_comum


## 7. Cálculo regional

In [7]:

def calcular_regional(grupo_comum, alvo, apoios_usados):
    # A estação alvo NÃO participa da média regional.
    regional = grupo_comum[list(apoios_usados)].mean(axis=1)
    regional.name = "precipitacao_regional"
    return regional


## 8. Acumuladas

In [8]:

def calcular_acumuladas(grupo_comum, alvo, regional):
    alvo_mensal = grupo_comum[str(alvo)].astype(float)
    regional_mensal = regional.astype(float)

    acumulado_alvo = alvo_mensal.cumsum()
    acumulado_regional = regional_mensal.cumsum()

    return alvo_mensal, regional_mensal, acumulado_alvo, acumulado_regional


## 9. Regressão

In [9]:

USAR_INTERCEPTO = bool(dm["regressao"].get("usar_intercepto", True))

def ajustar_regressao_dupla_massa(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) != len(y) or len(x) < 2:
        return {"ok": False, "motivo": "pontos_insuficientes_regressao"}

    if not np.isfinite(x).all() or not np.isfinite(y).all():
        return {"ok": False, "motivo": "valores_nao_finitos_regressao"}

    if np.isclose(np.var(x), 0.0):
        return {"ok": False, "motivo": "variancia_x_zero"}

    if USAR_INTERCEPTO:
        b, a = np.polyfit(x, y, 1)
    else:
        denom = float(np.dot(x, x))
        if np.isclose(denom, 0.0):
            return {"ok": False, "motivo": "denominador_zero"}
        b = float(np.dot(x, y) / denom)
        a = 0.0

    yhat = a + b * x
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))

    if np.isclose(ss_tot, 0.0):
        return {"ok": False, "motivo": "variancia_y_zero"}

    r2 = 1.0 - ss_res / ss_tot

    if not (np.isfinite(a) and np.isfinite(b) and np.isfinite(r2)):
        return {"ok": False, "motivo": "parametros_regressao_nao_finitos"}

    return {
        "ok": True,
        "motivo": "ok",
        "coeficiente_angular": float(b),
        "intercepto": float(a),
        "r2": float(r2),
        "y_ajustado": yhat,
    }

def formatar_equacao(b, a, casas=CASAS):
    sinal = "+" if a >= 0 else "-"
    return f"y = {b:.{casas}f}x {sinal} {abs(a):.{casas}f}"


## 10. R² e processamento completo por estação

In [10]:

def base_resumo(alvo):
    return {
        "estacao_alvo": str(alvo),
        "status": "nao_calculado",
        "motivo": None,
        "n_apoios_disponiveis": 0,
        "n_apoios_utilizados": 0,
        "estacoes_apoio_utilizadas": "",
        "rankings_apoio": "",
        "n_dados_apoio_1": np.nan,
        "n_dados_apoio_2": np.nan,
        "n_dados_apoio_3": np.nan,
        "n_meses_periodo_comum": 0,
        "data_inicial": pd.NaT,
        "data_final": pd.NaT,
        "coeficiente_angular": np.nan,
        "intercepto": np.nan,
        "r2_dupla_massa": np.nan,
        "equacao": "",
        "apoios_permitidos": 0,
        "apoios_inexistentes": "",
        "testes_status": "",
    }

def processar_estacao(alvo):
    resumo = base_resumo(alvo)

    # 1. alvo deve existir
    if str(alvo) not in dados.columns:
        resumo["motivo"] = "estacao_alvo_inexistente"
        return resumo, None, None

    sub_all = apoio.loc[apoio["estacao_alvo"].astype(str) == str(alvo)].copy()
    resumo["apoios_permitidos"] = int(len(sub_all))

    inexist = sorted(
        set(sub_all.loc[
            ~sub_all["estacao_apoio"].astype(str).isin(dados.columns),
            "estacao_apoio"
        ].astype(str))
    )
    resumo["apoios_inexistentes"] = "|".join(inexist)

    # 2. seleção
    selecao = selecionar_apoios(alvo)
    cand = selecao["candidatos"]
    sel = selecao["selecionados"]

    resumo["n_apoios_disponiveis"] = int(len(cand))

    if not selecao["ok"]:
        resumo["motivo"] = selecao["motivo"]
        return resumo, None, {"candidatos": cand, "selecionados": sel}

    apoios_usados = sel["estacao_apoio"].astype(str).tolist()
    ranks = sel["ranking"].astype(int).tolist()

    resumo["n_apoios_utilizados"] = len(apoios_usados)
    resumo["estacoes_apoio_utilizadas"] = "|".join(apoios_usados)
    resumo["rankings_apoio"] = "|".join(map(str, ranks))

    for i, n in enumerate(sel["n_dados_validos"].astype(int).tolist(), start=1):
        if i <= 3:
            resumo[f"n_dados_apoio_{i}"] = n

    # 3. período comum
    _, comum = construir_periodo_comum(alvo, apoios_usados)
    resumo["n_meses_periodo_comum"] = int(len(comum))

    if len(comum) < MIN_COMUM:
        resumo["motivo"] = "periodo_comum_insuficiente"
        return resumo, None, {"candidatos": cand, "selecionados": sel, "grupo_comum": comum}

    resumo["data_inicial"] = comum.index.min()
    resumo["data_final"] = comum.index.max()

    # Precipitação negativa tornaria acumuladas decrescentes.
    if dm["validacao"].get("exigir_precipitacao_nao_negativa", True):
        if (comum < 0).any().any():
            resumo["motivo"] = "precipitacao_negativa_periodo_comum"
            return resumo, None, {"candidatos": cand, "selecionados": sel, "grupo_comum": comum}

    # 4. regional e acumuladas
    regional = calcular_regional(comum, alvo, apoios_usados)
    alvo_m, reg_m, alvo_ac, reg_ac = calcular_acumuladas(comum, alvo, regional)

    # 5. regressão
    ajuste = ajustar_regressao_dupla_massa(reg_ac.values, alvo_ac.values)
    if not ajuste["ok"]:
        resumo["motivo"] = ajuste["motivo"]
        return resumo, None, {
            "candidatos": cand, "selecionados": sel, "grupo_comum": comum
        }

    b = ajuste["coeficiente_angular"]
    a = ajuste["intercepto"]
    r2 = ajuste["r2"]

    resumo.update({
        "status": "calculado",
        "motivo": "ok",
        "coeficiente_angular": b,
        "intercepto": a,
        "r2_dupla_massa": r2,
        "equacao": formatar_equacao(b, a),
    })

    detalhe = pd.DataFrame({
        "estacao_alvo": str(alvo),
        "data": comum.index,
        "n_apoios": len(apoios_usados),
        "estacoes_apoio": "|".join(apoios_usados),
        "precipitacao_alvo": alvo_m.values,
        "precipitacao_regional": reg_m.values,
        "acumulado_alvo": alvo_ac.values,
        "acumulado_regional": reg_ac.values,
    })

    contexto = {
        "candidatos": cand,
        "selecionados": sel,
        "grupo_comum": comum,
        "ajuste": ajuste,
        "apoios_usados": apoios_usados,
    }
    return resumo, detalhe, contexto


## 11. Tabela resumo

In [11]:

resumos = []
detalhes = []
contextos = {}

for alvo in alvos:
    resumo, detalhe, contexto = processar_estacao(alvo)
    resumos.append(resumo)
    contextos[str(alvo)] = contexto
    if detalhe is not None:
        detalhes.append(detalhe)

resumo_dupla_massa = pd.DataFrame(resumos)
dados_dupla_massa = (
    pd.concat(detalhes, ignore_index=True)
    if detalhes
    else pd.DataFrame(columns=[
        "estacao_alvo", "data", "n_apoios", "estacoes_apoio",
        "precipitacao_alvo", "precipitacao_regional",
        "acumulado_alvo", "acumulado_regional"
    ])
)

print("Situação das estações:")
display(resumo_dupla_massa["status"].value_counts(dropna=False).rename("n").to_frame())
display(
    resumo_dupla_massa[
        ["estacao_alvo", "estacoes_apoio_utilizadas",
         "n_meses_periodo_comum", "equacao", "r2_dupla_massa",
         "status", "motivo"]
    ].head(30)
)


Situação das estações:


,n
status,
calculado,19


,estacao_alvo,estacoes_apoio_utilizadas,n_meses_periodo_comum,equacao,r2_dupla_massa,status,motivo
0,2751007,2852046|2751017|2851043,536,y = 1.0103x + 7.6613,0.999856,calculado,ok
1,2751017,2751007|2852046|2851043,536,y = 1.0361x + 127.4185,0.999753,calculado,ok
2,2851003,2851024|2851021|2851022,540,y = 1.0360x + 663.7919,0.999843,calculado,ok
3,2851005,2851022|2851024|2851020,396,y = 0.9805x + 569.1068,0.999817,calculado,ok
4,2851020,2852046|2851022|2751007,526,y = 0.9860x - 366.6839,0.999749,calculado,ok
5,2851021,2851024|2851003|2851022,540,y = 0.8631x - 359.4199,0.999785,calculado,ok
6,2851022,2851024|2851003|2852004,540,y = 1.0462x + 440.3012,0.999695,calculado,ok
7,2851024,2851003|2851021|2851022,540,y = 1.0159x - 830.5224,0.999641,calculado,ok
8,2851028,2851022|2851024|2852046,499,y = 0.9406x - 134.9829,0.999612,calculado,ok
9,2851043,2751007|2852046|2751017,536,y = 0.9298x + 415.8849,0.999711,calculado,ok


## 12. Gráficos individuais

In [12]:

GRAF = dm["grafico"]

def titulo_estacao(row):
    alvo = row["estacao_alvo"]
    if row["status"] != "calculado":
        return f"Dupla massa — Estação {alvo}<br><sup>Status: não calculado | Motivo: {row['motivo']}</sup>"
    apoios_txt = row["estacoes_apoio_utilizadas"].replace("|", " | ")
    return (
        f"Dupla massa — Estação {alvo}"
        f"<br><sup>Apoios: {apoios_txt} | "
        f"Meses comuns: {int(row['n_meses_periodo_comum'])} | "
        f"R² = {row['r2_dupla_massa']:.{CASAS}f} | "
        f"{row['equacao']}</sup>"
    )

def adicionar_traces_estacao(fig, row, detalhe, visible=True):
    alvo = str(row["estacao_alvo"])
    if detalhe is None or detalhe.empty or row["status"] != "calculado":
        fig.add_trace(go.Scatter(
            x=[], y=[], mode="lines+markers",
            name=f"Dupla massa — {alvo}",
            visible=visible,
            hoverinfo="skip",
            showlegend=True,
        ))
        fig.add_trace(go.Scatter(
            x=[], y=[], mode="lines",
            name=f"Regressão — {alvo}",
            visible=visible,
            hoverinfo="skip",
            showlegend=True,
        ))
        if GRAF.get("mostrar_linha_referencia_origem", False):
            fig.add_trace(go.Scatter(
                x=[], y=[], mode="lines",
                name=f"Referência (0,0) — {alvo}",
                visible=visible,
                hoverinfo="skip",
                showlegend=True,
            ))
        return

    d = detalhe.sort_values("data").copy()

    customdata = np.column_stack([
        d["data"].dt.strftime("%Y-%m").to_numpy(),
        d["precipitacao_alvo"].to_numpy(dtype=float),
        d["precipitacao_regional"].to_numpy(dtype=float),
        d["acumulado_alvo"].to_numpy(dtype=float),
        d["acumulado_regional"].to_numpy(dtype=float),
    ])

    mode = "lines+markers" if (
        GRAF.get("mostrar_pontos", True)
        and GRAF.get("mostrar_linha_dupla_massa", True)
    ) else ("markers" if GRAF.get("mostrar_pontos", True) else "lines")

    fig.add_trace(go.Scatter(
        x=d["acumulado_regional"],
        y=d["acumulado_alvo"],
        mode=mode,
        name=f"Dupla massa — {alvo}",
        visible=visible,
        customdata=customdata,
        hovertemplate=(
            f"<b>Estação alvo: {alvo}</b><br>"
            "Data: %{customdata[0]}<br>"
            "Precipitação alvo mensal: %{customdata[1]:.3f}<br>"
            "Precipitação regional mensal: %{customdata[2]:.3f}<br>"
            "Acumulado alvo: %{customdata[3]:.3f}<br>"
            "Acumulado regional: %{customdata[4]:.3f}"
            "<extra></extra>"
        ),
    ))

    xline = np.array([0.0, float(d["acumulado_regional"].max())])
    yline = row["intercepto"] + row["coeficiente_angular"] * xline

    fig.add_trace(go.Scatter(
        x=xline,
        y=yline,
        mode="lines",
        name=f"Regressão — {alvo}",
        visible=visible and GRAF.get("mostrar_reta_regressao", True),
        hovertemplate=(
            f"<b>Regressão — {alvo}</b><br>"
            f"{row['equacao']}<br>"
            f"R² = {row['r2_dupla_massa']:.{CASAS}f}"
            "<extra></extra>"
        ),
    ))

    if GRAF.get("mostrar_linha_referencia_origem", False):
        xfinal = float(d["acumulado_regional"].iloc[-1])
        yfinal = float(d["acumulado_alvo"].iloc[-1])
        slope_ref = yfinal / xfinal if not np.isclose(xfinal, 0.0) else np.nan
        if np.isfinite(slope_ref):
            yref = slope_ref * xline
        else:
            yref = np.array([np.nan, np.nan])
        fig.add_trace(go.Scatter(
            x=xline,
            y=yref,
            mode="lines",
            name=f"Referência origem — {alvo}",
            visible=visible,
            line=dict(dash="dot"),
            hovertemplate=(
                "<b>Linha de referência pela origem</b><br>"
                f"Inclinação global = {slope_ref:.{CASAS}f}<br>"
                "Esta linha NÃO é a regressão."
                "<extra></extra>"
            ),
        ))

def figura_individual(row):
    alvo = str(row["estacao_alvo"])
    detalhe = (
        dados_dupla_massa.loc[dados_dupla_massa["estacao_alvo"] == alvo].copy()
        if not dados_dupla_massa.empty else pd.DataFrame()
    )
    fig = go.Figure()
    adicionar_traces_estacao(fig, row, detalhe, visible=True)
    fig.update_layout(
        template="plotly_white",
        height=int(GRAF.get("altura", 720)),
        title=titulo_estacao(row),
        xaxis_title="Precipitação regional acumulada",
        yaxis_title="Precipitação acumulada da estação alvo",
        hovermode="closest",
        legend_title="Séries",
    )
    return fig

if GRAF.get("salvar_individuais_html", True):
    n_salvos = 0
    for _, row in resumo_dupla_massa.loc[
        resumo_dupla_massa["status"] == "calculado"
    ].iterrows():
        fig = figura_individual(row)
        path = INDIV_DIR / f"dupla_massa_{row['estacao_alvo']}.html"
        fig.write_html(path, include_plotlyjs=True, full_html=True)
        n_salvos += 1
    print(f"Gráficos individuais HTML salvos: {n_salvos}")


Gráficos individuais HTML salvos: 19


## 13. Gráfico Plotly interativo com seletor de estação

In [13]:

fig_interativo = go.Figure()

n_traces_por_estacao = 3 if GRAF.get("mostrar_linha_referencia_origem", False) else 2

for i, (_, row) in enumerate(resumo_dupla_massa.iterrows()):
    alvo = str(row["estacao_alvo"])
    detalhe = (
        dados_dupla_massa.loc[dados_dupla_massa["estacao_alvo"] == alvo].copy()
        if not dados_dupla_massa.empty else pd.DataFrame()
    )
    adicionar_traces_estacao(fig_interativo, row, detalhe, visible=(i == 0))

buttons = []
n_est = len(resumo_dupla_massa)

for i, (_, row) in enumerate(resumo_dupla_massa.iterrows()):
    vis = [False] * (n_est * n_traces_por_estacao)
    ini = i * n_traces_por_estacao
    for j in range(n_traces_por_estacao):
        # A função adiciona reta de regressão como trace mesmo quando a opção
        # de exibição está false; nesse caso ela fica invisível.
        if j == 1 and not GRAF.get("mostrar_reta_regressao", True):
            vis[ini + j] = False
        else:
            vis[ini + j] = True

    buttons.append(dict(
        label=str(row["estacao_alvo"]),
        method="update",
        args=[
            {"visible": vis},
            {"title": {"text": titulo_estacao(row)}}
        ],
    ))

titulo_inicial = (
    titulo_estacao(resumo_dupla_massa.iloc[0])
    if len(resumo_dupla_massa)
    else "Dupla massa — nenhuma estação alvo"
)

fig_interativo.update_layout(
    template="plotly_white",
    height=int(GRAF.get("altura", 720)),
    title=titulo_inicial,
    xaxis_title="Precipitação regional acumulada",
    yaxis_title="Precipitação acumulada da estação alvo",
    hovermode="closest",
    legend_title="Séries",
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True,
        x=0.0,
        xanchor="left",
        y=1.18,
        yanchor="top",
    )] if buttons and GRAF.get("dropdown_estacoes", True) else [],
    annotations=[
        dict(
            text="Selecionar estação:",
            x=0.0, y=1.245,
            xref="paper", yref="paper",
            showarrow=False,
            xanchor="left",
        )
    ] if buttons and GRAF.get("dropdown_estacoes", True) else [],
)

print(f"Estações disponíveis no seletor: {len(buttons)}")


Estações disponíveis no seletor: 19


## 14. Exportação

In [14]:

arquivo_resumo = OUTPUT_DIR / OUT_CFG["resumo"]
arquivo_dados = OUTPUT_DIR / OUT_CFG["dados_detalhados"]
arquivo_testes = OUTPUT_DIR / OUT_CFG["testes"]
arquivo_interativo = OUTPUT_DIR / OUT_CFG["interativo_html"]
arquivo_cfg = OUTPUT_DIR / OUT_CFG["config_utilizada"]

# As datas são exportadas como YYYY-MM.
resumo_export = resumo_dupla_massa.copy()
for c in ["data_inicial", "data_final"]:
    resumo_export[c] = pd.to_datetime(
        resumo_export[c], errors="coerce"
    ).dt.strftime("%Y-%m")

dados_export = dados_dupla_massa.copy()
if not dados_export.empty:
    dados_export["data"] = pd.to_datetime(dados_export["data"]).dt.strftime("%Y-%m")

resumo_export.to_csv(
    arquivo_resumo, sep=SEP, decimal=DECIMAL, encoding=ENCODING,
    index=False, na_rep=REP_NAN
)
dados_export.to_csv(
    arquivo_dados, sep=SEP, decimal=DECIMAL, encoding=ENCODING,
    index=False, na_rep=REP_NAN
)

fig_interativo.write_html(
    arquivo_interativo,
    include_plotlyjs=True,
    full_html=True
)

shutil.copy2(CONFIG_PATH, arquivo_cfg)

print("Exportados:")
for p in [arquivo_resumo, arquivo_dados, arquivo_interativo, arquivo_cfg]:
    print(" -", p)


Exportados:
 - A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\dupla_massa\resumo_dupla_massa.csv
 - A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\dupla_massa\dados_dupla_massa.csv
 - A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\dupla_massa\dupla_massa_interativo.html
 - A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\dupla_massa\config_dupla_massa_utilizada.yml


## 15. Testes de consistência

In [15]:

testes = []

def add_teste(alvo, numero, nome, ok, detalhe=""):
    testes.append({
        "estacao_alvo": str(alvo),
        "teste": numero,
        "nome": nome,
        "resultado": "OK" if ok else "FALHOU",
        "detalhe": detalhe,
    })
    return bool(ok)

for _, row in resumo_dupla_massa.iterrows():
    alvo = str(row["estacao_alvo"])
    contexto = contextos.get(alvo)
    calculado = row["status"] == "calculado"

    # Teste 1
    t1 = alvo in dados.columns
    add_teste(alvo, 1, "Estação alvo existe no arquivo de precipitação", t1)

    if not calculado:
        # Para estações não calculadas, os demais testes operacionais são registrados
        # como não aplicáveis, sem interromper o processamento de outras estações.
        for num, nome in [
            (2, "Apoios selecionados existem no arquivo"),
            (3, "Nenhum apoio é igual ao alvo"),
            (4, f"No máximo {MAX_APOIOS} apoios utilizados"),
            (5, f"No mínimo {MIN_APOIOS} apoio utilizado"),
            (6, "Selecionadas as maiores disponibilidades entre apoios permitidos"),
            (7, "Período comum sem NaN"),
            (8, "Alvo e apoios usam exatamente as mesmas datas"),
            (9, "Dados em ordem cronológica"),
            (10, "Acumulado regional não diminui"),
            (11, "Acumulado do alvo não diminui"),
            (12, "X e Y possuem o mesmo número de pontos"),
            (13, "R² finito"),
            (14, "Coeficiente angular e intercepto finitos"),
            (15, "Gráfico corresponde aos dados exportados"),
        ]:
            testes.append({
                "estacao_alvo": alvo,
                "teste": num,
                "nome": nome,
                "resultado": "NAO_APLICAVEL",
                "detalhe": f"status=nao_calculado; motivo={row['motivo']}",
            })
        continue

    sel = contexto["selecionados"].copy()
    cand = contexto["candidatos"].copy()
    comum = contexto["grupo_comum"].copy()
    apoios_usados = contexto["apoios_usados"]
    d = dados_dupla_massa.loc[dados_dupla_massa["estacao_alvo"] == alvo].sort_values("data")

    t2 = all(e in dados.columns for e in apoios_usados)
    add_teste(alvo, 2, "Apoios selecionados existem no arquivo", t2)

    t3 = all(e != alvo for e in apoios_usados)
    add_teste(alvo, 3, "Nenhum apoio é igual ao alvo", t3)

    t4 = len(apoios_usados) <= MAX_APOIOS
    add_teste(alvo, 4, f"No máximo {MAX_APOIOS} apoios utilizados", t4)

    t5 = len(apoios_usados) >= MIN_APOIOS
    add_teste(alvo, 5, f"No mínimo {MIN_APOIOS} apoio utilizado", t5)

    # A seleção deve ser exatamente os primeiros MAX_APOIOS após o sort definido.
    esperado = cand.head(MAX_APOIOS)["estacao_apoio"].astype(str).tolist()
    t6 = apoios_usados == esperado
    add_teste(
        alvo, 6,
        "Selecionadas as maiores disponibilidades entre apoios permitidos",
        t6,
        f"selecionados={apoios_usados}; esperado={esperado}"
    )

    t7 = not comum.isna().any().any()
    add_teste(alvo, 7, "Período comum sem NaN", t7)

    # Todos pertencem ao mesmo DataFrame comum, portanto as datas são idênticas.
    t8 = all(comum[c].index.equals(comum.index) for c in [alvo] + apoios_usados)
    add_teste(alvo, 8, "Alvo e apoios usam exatamente as mesmas datas", t8)

    t9 = comum.index.is_monotonic_increasing and d["data"].is_monotonic_increasing
    add_teste(alvo, 9, "Dados em ordem cronológica", t9)

    t10 = bool((d["acumulado_regional"].diff().dropna() >= -1e-12).all())
    add_teste(alvo, 10, "Acumulado regional não diminui", t10)

    t11 = bool((d["acumulado_alvo"].diff().dropna() >= -1e-12).all())
    add_teste(alvo, 11, "Acumulado do alvo não diminui", t11)

    t12 = len(d["acumulado_regional"]) == len(d["acumulado_alvo"])
    add_teste(alvo, 12, "X e Y possuem o mesmo número de pontos", t12)

    t13 = np.isfinite(float(row["r2_dupla_massa"]))
    add_teste(alvo, 13, "R² finito", t13)

    t14 = (
        np.isfinite(float(row["coeficiente_angular"]))
        and np.isfinite(float(row["intercepto"]))
    )
    add_teste(alvo, 14, "Coeficiente angular e intercepto finitos", t14)

    # Reconstrói a figura individual e compara a trace observada com os dados exportáveis.
    fig_check = figura_individual(row)
    trace_obs = fig_check.data[0]
    x_plot = np.asarray(trace_obs.x, dtype=float)
    y_plot = np.asarray(trace_obs.y, dtype=float)
    x_csv = d["acumulado_regional"].to_numpy(dtype=float)
    y_csv = d["acumulado_alvo"].to_numpy(dtype=float)
    t15 = (
        len(x_plot) == len(x_csv)
        and len(y_plot) == len(y_csv)
        and np.allclose(x_plot, x_csv, equal_nan=False)
        and np.allclose(y_plot, y_csv, equal_nan=False)
    )
    add_teste(alvo, 15, "Gráfico corresponde aos dados exportados", t15)

testes_dupla_massa = pd.DataFrame(testes)

# Atualiza resumo com status dos testes por estação.
if not testes_dupla_massa.empty:
    status_testes = {}
    for alvo, grp in testes_dupla_massa.groupby("estacao_alvo"):
        falhas = grp.loc[grp["resultado"] == "FALHOU"]
        status_testes[str(alvo)] = "OK" if falhas.empty else f"FALHOU_{len(falhas)}"
    resumo_dupla_massa["testes_status"] = resumo_dupla_massa["estacao_alvo"].map(status_testes)

# Falha em teste de uma estação calculada não interrompe as demais, mas fica explícita.
falhas_testes = testes_dupla_massa.loc[testes_dupla_massa["resultado"] == "FALHOU"]

testes_dupla_massa.to_csv(
    arquivo_testes, sep=SEP, decimal=DECIMAL, encoding=ENCODING,
    index=False, na_rep=REP_NAN
)

# Reexporta o resumo final com a coluna testes_status atualizada.
resumo_export = resumo_dupla_massa.copy()
for c in ["data_inicial", "data_final"]:
    resumo_export[c] = pd.to_datetime(
        resumo_export[c], errors="coerce"
    ).dt.strftime("%Y-%m")
resumo_export.to_csv(
    arquivo_resumo, sep=SEP, decimal=DECIMAL, encoding=ENCODING,
    index=False, na_rep=REP_NAN
)

print("Resumo dos testes:")
display(testes_dupla_massa["resultado"].value_counts(dropna=False).rename("n").to_frame())

if not falhas_testes.empty:
    warnings.warn(
        f"{len(falhas_testes)} teste(s) falharam. Consulte {arquivo_testes}. "
        "As demais estações continuaram sendo processadas."
    )

print("\nTabela final:")
display(
    resumo_dupla_massa[
        ["estacao_alvo", "estacoes_apoio_utilizadas",
         "n_meses_periodo_comum", "equacao", "r2_dupla_massa",
         "status", "motivo", "testes_status"]
    ]
)

if GRAF.get("exibir_interativo_no_notebook", True):
    fig_interativo.show()


Resumo dos testes:


,n
resultado,
OK,285



Tabela final:


,estacao_alvo,estacoes_apoio_utilizadas,n_meses_periodo_comum,equacao,r2_dupla_massa,status,motivo,testes_status
0,2751007,2852046|2751017|2851043,536,y = 1.0103x + 7.6613,0.999856,calculado,ok,OK
1,2751017,2751007|2852046|2851043,536,y = 1.0361x + 127.4185,0.999753,calculado,ok,OK
2,2851003,2851024|2851021|2851022,540,y = 1.0360x + 663.7919,0.999843,calculado,ok,OK
3,2851005,2851022|2851024|2851020,396,y = 0.9805x + 569.1068,0.999817,calculado,ok,OK
4,2851020,2852046|2851022|2751007,526,y = 0.9860x - 366.6839,0.999749,calculado,ok,OK
5,2851021,2851024|2851003|2851022,540,y = 0.8631x - 359.4199,0.999785,calculado,ok,OK
6,2851022,2851024|2851003|2852004,540,y = 1.0462x + 440.3012,0.999695,calculado,ok,OK
7,2851024,2851003|2851021|2851022,540,y = 1.0159x - 830.5224,0.999641,calculado,ok,OK
8,2851028,2851022|2851024|2852046,499,y = 0.9406x - 134.9829,0.999612,calculado,ok,OK
9,2851043,2751007|2852046|2751017,536,y = 0.9298x + 415.8849,0.999711,calculado,ok,OK
